# Random Forest Training Notebook
This notebook trains a Random Forest model for energy consumption prediction using the cloudscheduling_cleaned.csv dataset and displays comprehensive accuracy metrics including R², RMSE, MAE, Accuracy %, and feature importance.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
import json

print('Libraries imported successfully')

Libraries imported successfully


In [2]:
# Load data from CSV
csv_path = 'cloudscheduling_cleaned.csv'
print(f"Loading data from: {csv_path}")

df = pd.read_csv(csv_path)
print(f"Dataset loaded: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

# Prepare features and target
target_col = 'Execution_Time (s)'
columns_to_drop = ['Task_ID', target_col]

# Remove columns that don't exist
columns_to_drop = [col for col in columns_to_drop if col in df.columns]

X = df.drop(columns=columns_to_drop)
y = df[target_col]

print(f"\nFeatures: {list(X.columns)}")
print(f"Target: {target_col}")
print(f"\nTarget statistics:")
print(f"  Mean: {y.mean():.2f}")
print(f"  Std: {y.std():.2f}")
print(f"  Range: {y.min():.2f} - {y.max():.2f}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining samples: {len(X_train)}, Test samples: {len(X_test)}")

Loading data from: cloudscheduling_cleaned.csv
Dataset loaded: (20000, 9)

Columns: ['Task_ID', 'CPU_Usage (%)', 'RAM_Usage (MB)', 'Disk_IO (MB/s)', 'Network_IO (MB/s)', 'Priority', 'VM_ID', 'Execution_Time (s)', 'Target (Optimal Scheduling)']

First few rows:
   Task_ID  CPU_Usage (%)  RAM_Usage (MB)  Disk_IO (MB/s)  Network_IO (MB/s)  \
0  0.00000       0.078125        0.274211        0.648936           0.270833   
1  0.00005       0.421875        0.635121        0.840426           0.583333   
2  0.00010       0.500000        0.508349        0.340426           0.354167   
3  0.00015       0.453125        0.194254        0.872340           0.645833   
4  0.00020       0.734375        0.582257        0.393617           0.708333   

   Priority     VM_ID  Execution_Time (s)  Target (Optimal Scheduling)  
0       0.0  0.333333            0.623333                          0.0  
1       1.0  0.888889            0.456667                          0.0  
2       0.5  0.000000            0.7166

In [3]:
print("Training Random Forest model...")

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

print("\nTraining completed!")

Training Random Forest model...

Training completed!


In [4]:
# Evaluate model
y_pred = rf.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Cross-validation
cv_scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='r2')

# Calculate accuracy as percentage (based on mean absolute percentage error)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
accuracy = 100 - mape

# Feature importance
feature_importance = dict(zip(X_train.columns, rf.feature_importances_))

print('\n' + '='*50)
print('RANDOM FOREST EVALUATION RESULTS')
print('='*50)
print(f'R² Score: {r2:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'MAE: {mae:.4f}')
print(f'MSE: {mse:.4f}')
print(f'Accuracy: {accuracy:.2f}%')
print(f'Cross-validation R² mean: {cv_scores.mean():.4f}')
print(f'Cross-validation R² std: {cv_scores.std():.4f}')
print('='*50)

print('\nTop Feature Importances:')
for feat, val in sorted(feature_importance.items(), key=lambda x: x[1], reverse=True):
    print(f'  {feat}: {val:.4f}')

results = {
    'r2': float(r2),
    'rmse': float(rmse),
    'mae': float(mae),
    'mse': float(mse),
    'accuracy_percent': float(accuracy),
    'cv_mean': float(cv_scores.mean()),
    'cv_std': float(cv_scores.std()),
    'feature_importance': {k: float(v) for k, v in feature_importance.items()}
}


RANDOM FOREST EVALUATION RESULTS
R² Score: 0.7387
RMSE: 0.1478
MAE: 0.1267
MSE: 0.0218
Accuracy: -inf%
Cross-validation R² mean: 0.7373
Cross-validation R² std: 0.0052

Top Feature Importances:
  Target (Optimal Scheduling): 0.8780
  RAM_Usage (MB): 0.0319
  Disk_IO (MB/s): 0.0258
  CPU_Usage (%): 0.0236
  Network_IO (MB/s): 0.0220
  VM_ID: 0.0132
  Priority: 0.0055
